# Data Visualization of Banned Books

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import string
import re

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

# the 'magic' or the percent symbol means it'll look good visually

In [2]:
data = pd.read_csv("openlibrary_subjects_match.csv")
data.head()

,clean_title,clean_author,subjects,source
0,...And Their Memory Was A Bitter Tree,Robert Howard,"['Fiction, fantasy, short stories']",openlibrary.org
1,(Non)Conform Russian And Soviet Art 1958-1995,Barbara Thiemann,[],openlibrary.org
2,Scuse Me While I Kiss The Sky,David Henderson,[],openlibrary.org
3,A Black Gaze,Tina Campt,[],openlibrary.org
4,A Black Rose Thrived,Rochelle Richey,[],openlibrary.org


In [3]:
data = data[data.subjects != "[]"] # drop all rows without subjects

In [4]:
data.head()

,clean_title,clean_author,subjects,source
0,...And Their Memory Was A Bitter Tree,Robert Howard,"['Fiction, fantasy, short stories']",openlibrary.org
5,A Brief History Of Album Covers,Jason Draper,"['Sound recordings', 'Packaging']",openlibrary.org
7,A Brief History Of Vice,Robert Evans,"['Vice', 'Modern Civilization', 'History', 'Ci...",openlibrary.org
9,A Centaur'S Life 10,Kei Murayama,"['Centaurs', 'Comic books, strips', 'High scho...",openlibrary.org
14,A Centaur'S Life 2,Kei Murayama,"['Juvenile fiction', 'Graphic novels', 'High s...",openlibrary.org


In [5]:
data.shape

(3775, 4)

In [6]:
banned_books = pd.read_csv("data/banned_book_data_tx-list.csv")
banned_books.head()

,publication,author,date,unit_deny_reason,reason,"note_on_sourcing:_i_got_this_information_through_a_request_to_the_tdcj_public_information_office,_received_10/14/2022.",year,month,day,state_arc
0,...AND THEIR MEMORY WAS A BITTER TREE,"HOWARD, ROBERT",2016-08-19,PAGES 81 & 369 SEXUALLY EXPLICIT IMAGE,PGS 81 & 369 SEXUALLY EXPLICIT IMAGE,NaN,2016,8,19,tx
1,"(NON)CONFORM RUSSIAN AND SOVIET ART 1958-1995,","THIEMANN, BARBARA",2012-09-10,PAGE 422 PHOTO OF A NUDE CHILD,PG 422 NUDE CHILD,NaN,2012,9,10,tx
2,SCUSE ME WHILE I KISS THE SKY,"HENDERSON, DAVID",2019-07-17,PG 294 SEXUALLY EXPLICIT IMAGE,PG 294 SEXUALLY EXPLICIT IMAGE,NaN,2019,7,17,tx
3,SCUSE ME WHILE I KISS THE SKY,"HENDERSON, DAVID",2010-01-29,PAGE 310 CONTAINS SEXUALLY EXPLICIT IMAGES,PAGE 9 OF PHOTO INSERT CONTAINS SEXUALLY EXPLI...,NaN,2010,1,29,tx
4,A BLACK GAZE,"CAMPT, TINA",2021-09-08,"PAGES 28, 30, 35 SEXUALLY EXPLICIT IMAGES","PAGES 28, 30 & 35 CONTAIN SEXUALLY EXPLICIT IM...",NaN,2021,9,8,tx


In [7]:
# clean up column names
banned_books = banned_books.rename(columns={
    "publication": "publication",
    "author": "author",
    "date": "date",
    "unit_deny_reason": "unit_reason",
    "reason": "reason",
    "year": "year",
    "month": "month",
    "day": "day",
    "state_arc": "state"
})

banned_books = banned_books.drop(columns=["note_on_sourcing:_i_got_this_information_through_a_request_to_the_tdcj_public_information_office,_received_10/14/2022."], errors="ignore")
banned_books.head()

,publication,author,date,unit_reason,reason,year,month,day,state
0,...AND THEIR MEMORY WAS A BITTER TREE,"HOWARD, ROBERT",2016-08-19,PAGES 81 & 369 SEXUALLY EXPLICIT IMAGE,PGS 81 & 369 SEXUALLY EXPLICIT IMAGE,2016,8,19,tx
1,"(NON)CONFORM RUSSIAN AND SOVIET ART 1958-1995,","THIEMANN, BARBARA",2012-09-10,PAGE 422 PHOTO OF A NUDE CHILD,PG 422 NUDE CHILD,2012,9,10,tx
2,SCUSE ME WHILE I KISS THE SKY,"HENDERSON, DAVID",2019-07-17,PG 294 SEXUALLY EXPLICIT IMAGE,PG 294 SEXUALLY EXPLICIT IMAGE,2019,7,17,tx
3,SCUSE ME WHILE I KISS THE SKY,"HENDERSON, DAVID",2010-01-29,PAGE 310 CONTAINS SEXUALLY EXPLICIT IMAGES,PAGE 9 OF PHOTO INSERT CONTAINS SEXUALLY EXPLI...,2010,1,29,tx
4,A BLACK GAZE,"CAMPT, TINA",2021-09-08,"PAGES 28, 30, 35 SEXUALLY EXPLICIT IMAGES","PAGES 28, 30 & 35 CONTAIN SEXUALLY EXPLICIT IM...",2021,9,8,tx


In [8]:
# clean the data
## step 1: convert to lowercase
banned_books["clean_reason"] = (
    banned_books["reason"]
    .str.lower()
)

banned_books["clean_reason"].head()

0                 pgs 81 & 369 sexually explicit image
1                                    pg 422 nude child
2                       pg 294 sexually explicit image
3    page 9 of photo insert contains sexually expli...
4    pages 28, 30 & 35 contain sexually explicit im...
Name: clean_reason, dtype: object

In [9]:
# clean the data
## step 2: remove punctuation
banned_books["clean_reason"] = (
    banned_books["clean_reason"]
    .str.translate(
        str.maketrans("", "", string.punctuation)
    )
)

banned_books["clean_reason"].head()

0                  pgs 81  369 sexually explicit image
1                                    pg 422 nude child
2                       pg 294 sexually explicit image
3    page 9 of photo insert contains sexually expli...
4     pages 28 30  35 contain sexually explicit images
Name: clean_reason, dtype: object

In [10]:
# clean the data
## step 3: remove 'pgs' and numbers
banned_books["clean_reason"] = (
    banned_books["clean_reason"]
    .str.replace(r"\bpgs\b", "", regex = True) # remove pgs
    .str.replace(r"\bpg\b", "", regex = True) # remove pg
    .str.replace(r"\bpage\b", "", regex = True) # remove page
    .str.replace(r"\bpages\b", "", regex = True) # remove pages
    .str.replace(r"\d+", "", regex = True) # remove numbers
    .str.strip()
)

banned_books["clean_reason"].head(100)

0                               sexually explicit image
1                                            nude child
2                               sexually explicit image
3     of photo insert contains sexually explicit images
4                      contain sexually explicit images
                            ...                        
95                                     sex with a minor
96                                incest brotherbrother
97                                                 rape
98                                                 rape
99                                                 rape
Name: clean_reason, Length: 100, dtype: object

In [11]:
# clean the data: title format
# lets define a function 'clean_title'
def clean_title(title):
    if pd.isna(title):
        return None

    title = str(title).strip()

    # re.sub(pattern, repl, string, count=0, flags=0)
    # return the string obtained by replacing the left-most non-overlapping
        # occurences of pattern in string by the replacement repl
            # r = raw string
            # , = ,
            # $ = if at the end of the string
        # en gros: replace trailing commas with nothing
    title = re.sub(r',$', '', title)

    # collapse spaces (solves problem of accidental double spacing)
    # \s = any whitespace character i.e. " " "\t" "\n"
    # + = one or more in a row
    title = re.sub(r'\s+', ' ', title)

    return title

In [12]:
# clean the data: author format
# let's define a function 'author_first_last'
# the .strip() function removes leading and trailing spaces/characters
    #(depending on the argument) to return a copy of the original string

def author_first_last(author):
    
    if pd.isna(author):
        return None

    author = author.strip() # default is spaces

    if "," in author:
        last, first = author.split(",", 1) # using , as the delimiter
        return f"{first.strip()} {last.strip()}" # recombine first & last name

    return author

In [13]:
banned_books["clean_author"] = (
    banned_books["author"]
    .apply(author_first_last)
    .str.title()
)

In [14]:
# banned_books.head()

In [15]:
banned_books["clean_title"] = (
    banned_books["publication"]
    .apply(clean_title)
    .str.title()
)

In [16]:
banned_books.head()

,publication,author,date,unit_reason,reason,year,month,day,state,clean_reason,clean_author,clean_title
0,...AND THEIR MEMORY WAS A BITTER TREE,"HOWARD, ROBERT",2016-08-19,PAGES 81 & 369 SEXUALLY EXPLICIT IMAGE,PGS 81 & 369 SEXUALLY EXPLICIT IMAGE,2016,8,19,tx,sexually explicit image,Robert Howard,...And Their Memory Was A Bitter Tree
1,"(NON)CONFORM RUSSIAN AND SOVIET ART 1958-1995,","THIEMANN, BARBARA",2012-09-10,PAGE 422 PHOTO OF A NUDE CHILD,PG 422 NUDE CHILD,2012,9,10,tx,nude child,Barbara Thiemann,(Non)Conform Russian And Soviet Art 1958-1995
2,SCUSE ME WHILE I KISS THE SKY,"HENDERSON, DAVID",2019-07-17,PG 294 SEXUALLY EXPLICIT IMAGE,PG 294 SEXUALLY EXPLICIT IMAGE,2019,7,17,tx,sexually explicit image,David Henderson,Scuse Me While I Kiss The Sky
3,SCUSE ME WHILE I KISS THE SKY,"HENDERSON, DAVID",2010-01-29,PAGE 310 CONTAINS SEXUALLY EXPLICIT IMAGES,PAGE 9 OF PHOTO INSERT CONTAINS SEXUALLY EXPLI...,2010,1,29,tx,of photo insert contains sexually explicit images,David Henderson,Scuse Me While I Kiss The Sky
4,A BLACK GAZE,"CAMPT, TINA",2021-09-08,"PAGES 28, 30, 35 SEXUALLY EXPLICIT IMAGES","PAGES 28, 30 & 35 CONTAIN SEXUALLY EXPLICIT IM...",2021,9,8,tx,contain sexually explicit images,Tina Campt,A Black Gaze


In [20]:
merged = pd.merge(
    banned_books,
    data[["subjects", "source", "clean_author", "clean_title"]],
    on=["clean_author","clean_title"],
    how="left"
)

merged.head()

,publication,author,date,unit_reason,reason,year,month,day,state,clean_reason,clean_author,clean_title,subjects,source
0,...AND THEIR MEMORY WAS A BITTER TREE,"HOWARD, ROBERT",2016-08-19,PAGES 81 & 369 SEXUALLY EXPLICIT IMAGE,PGS 81 & 369 SEXUALLY EXPLICIT IMAGE,2016,8,19,tx,sexually explicit image,Robert Howard,...And Their Memory Was A Bitter Tree,"['Fiction, fantasy, short stories']",openlibrary.org
1,"(NON)CONFORM RUSSIAN AND SOVIET ART 1958-1995,","THIEMANN, BARBARA",2012-09-10,PAGE 422 PHOTO OF A NUDE CHILD,PG 422 NUDE CHILD,2012,9,10,tx,nude child,Barbara Thiemann,(Non)Conform Russian And Soviet Art 1958-1995,NaN,NaN
2,SCUSE ME WHILE I KISS THE SKY,"HENDERSON, DAVID",2019-07-17,PG 294 SEXUALLY EXPLICIT IMAGE,PG 294 SEXUALLY EXPLICIT IMAGE,2019,7,17,tx,sexually explicit image,David Henderson,Scuse Me While I Kiss The Sky,NaN,NaN
3,SCUSE ME WHILE I KISS THE SKY,"HENDERSON, DAVID",2010-01-29,PAGE 310 CONTAINS SEXUALLY EXPLICIT IMAGES,PAGE 9 OF PHOTO INSERT CONTAINS SEXUALLY EXPLI...,2010,1,29,tx,of photo insert contains sexually explicit images,David Henderson,Scuse Me While I Kiss The Sky,NaN,NaN
4,A BLACK GAZE,"CAMPT, TINA",2021-09-08,"PAGES 28, 30, 35 SEXUALLY EXPLICIT IMAGES","PAGES 28, 30 & 35 CONTAIN SEXUALLY EXPLICIT IM...",2021,9,8,tx,contain sexually explicit images,Tina Campt,A Black Gaze,NaN,NaN


In [22]:
merged = merged.drop(columns=["publication","author","unit_reason","reason","month","day"], errors="ignore")
merged.head()

,date,year,state,clean_reason,clean_author,clean_title,subjects,source
0,2016-08-19,2016,tx,sexually explicit image,Robert Howard,...And Their Memory Was A Bitter Tree,"['Fiction, fantasy, short stories']",openlibrary.org
1,2012-09-10,2012,tx,nude child,Barbara Thiemann,(Non)Conform Russian And Soviet Art 1958-1995,NaN,NaN
2,2019-07-17,2019,tx,sexually explicit image,David Henderson,Scuse Me While I Kiss The Sky,NaN,NaN
3,2010-01-29,2010,tx,of photo insert contains sexually explicit images,David Henderson,Scuse Me While I Kiss The Sky,NaN,NaN
4,2021-09-08,2021,tx,contain sexually explicit images,Tina Campt,A Black Gaze,NaN,NaN
